## Gemini Generation

In [1]:
import os
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import random

#API Keys are required to run the generation scripts but have been removed for security purposes.
client = genai.Client(api_key="Your_API_Key") 

PROMPT_BASE = "A fully unique and highly randomized real image of a place. Use a different categories, completely different pose, different camera lens, different environment, different lighting, and different composition every time. No template reuse. No similarities to previous images. Dramatically varied visuals."
NUM_IMAGES = 50
OUTPUT_FOLDER = "gemini_place_images"

def create_and_save_images():
    """สร้างรูปภาพเสมือนจริงของคนตามจำนวนที่กำหนด แล้วบันทึกในโฟลเดอร์"""
    
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    print(f"✅ โฟลเดอร์ '{OUTPUT_FOLDER}' พร้อมใช้งานแล้ว")

place_category = [
    "urban place",
    "natural place",
    "indoor place",
    "cultural place",
    "historical place",
    "commercial place",
    "transportation place",
    "educational place",
    "recreational place"
]

style = [
    "minimalist",
    "modern",
    "luxury",
    "vintage",
    "industrial",
    "futuristic",
    "clean aesthetic",
]

for i in range(1, NUM_IMAGES + 1):
    # random keyword from each groups
    selected_cat = random.choice(place_category)
    selected_style = random.choice(style)

    # random seed number
    random_seed = random.randint(1, 9999999)
    
    # combine prompt
    prompt_full = f"{PROMPT_BASE} , {selected_cat}, {selected_style} --seed {random_seed}"
        
    file_name = f"gemini_place_photorealistic_{str(i).zfill(3)}.jpg"
    output_path = os.path.join(OUTPUT_FOLDER, file_name)
        
    print(f"⏳ กำลังสร้างรูปภาพที่ {i}/{NUM_IMAGES}: '{file_name}'...")

    try:
            # เรียกใช้ Model เพื่อสร้างรูปภาพ
            response = client.models.generate_content(
                model="gemini-2.5-flash-image",
                contents=[prompt_full],
                # สามารถเพิ่ม image_generation_config เพื่อกำหนดสไตล์, อัตราส่วน ฯลฯ ได้
                # config=types.GenerateContentConfig(
                #     image_generation_config=types.ImageGenerationConfig(
                #         number_of_images=1,
                #         aspect_ratio="1:1"
                #     )
                # )
            )

            # 5. save picture
            for part in response.parts:
                if part.inline_data is not None:
                    # แปลงข้อมูลไบนารีเป็นวัตถุ Image
                    image_data = part.inline_data.data
                    image = Image.open(BytesIO(image_data))
                    
                    # บันทึกรูปภาพในรูปแบบ JPEG
                    image.save(output_path, format="JPEG")
                    print(f"  ✨ บันทึกสำเร็จ: {output_path}")
                    break
        
    except Exception as e:
            print(f"  ❌ เกิดข้อผิดพลาดในการสร้าง/บันทึกรูปภาพที่ {i}: {e}")
            # หากเกิดข้อผิดพลาด ให้ข้ามไปสร้างรูปถัดไป

    print(f"\n🎉 การสร้างรูปภาพเสร็จสมบูรณ์ 50 รูป ถูกเก็บไว้ในโฟลเดอร์: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    create_and_save_images()

⏳ กำลังสร้างรูปภาพที่ 1/50: 'gemini_place_photorealistic_001.jpg'...


KeyboardInterrupt: 

## Seadream Generation

In [ ]:
import os
import requests # จำเป็นสำหรับการดาวน์โหลดรูปภาพจาก URL
from openai import OpenAI
from io import BytesIO # ไม่ได้ใช้ แต่เก็บไว้เผื่อในอนาคต
import random

# --- การตั้งค่า Client ---
# *** คำเตือน: คุณได้ใส่ API Key แบบ hardcode ในโค้ดเดิม (d961f746-cabc-4733-92a8-1940f8e52ef8)
# ซึ่งไม่แนะนำด้านความปลอดภัย แต่ผมจะคงโครงสร้างเดิมไว้ หากต้องการความปลอดภัยให้ใช้ os.environ.get("ARK_API_KEY")

API_KEY = "5e9ddaed-9404-4f53-8a99-db09a6592c2a"
BASE_URL = "https://ark.ap-southeast.bytepluses.com/api/v3"
MODEL_NAME = "seedream-4-5-251128"

# Initialize the Ark client (using OpenAI SDK compatibility)
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)

# --- การตั้งค่าการสร้างรูปภาพ ---
PROMPT_BASE = "A photorealistic portrait of food."

# จำนวนรูปภาพที่ต้องการสร้าง
NUM_IMAGES = 50

# ชื่อโฟลเดอร์สำหรับจัดเก็บรูปภาพ
OUTPUT_FOLDER = "seedream_people_images"
# -----------------------------

def create_and_save_images():
    """สร้างรูปภาพเสมือนจริงของคนตามจำนวนที่กำหนด แล้วบันทึกในโฟลเดอร์"""
    
    # 1. สร้างโฟลเดอร์หากยังไม่มี
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    print(f"✅ โฟลเดอร์ '{OUTPUT_FOLDER}' พร้อมใช้งานแล้ว")

# 1. เตรียม Lists ของความหลากหลาย
food_category = [
    "main dish",
    "street food",
    "dessert",
    "bakery item",
    "snack",
    "beverage"
]

dish_type = [
    "grilled steak",
    "fried chicken",
    "ramen bowl",
    "pad thai",
    "cheeseburger",
    "chocolate cake",
    "croissant",
    "donut",
    "french fries",
    "iced coffee"
]

camera_angle = [
    "close-up food shot",
    "overhead flat lay",
    "45-degree angle",
    "eye-level shot",
    "macro food shot"
]


lighting = [
    "natural sunlight",
    "soft daylight",
    "studio lighting",
    "cinematic lighting",
    "warm indoor lighting"
]

presentation = [
    "plated elegantly",
    "rustic presentation",
    "street-style serving",
    "minimalist plating",
    "restaurant-style presentation"
]

realism_style = [
    "photorealistic food photography",
    "high detail realism",
    "natural color grading",
    "professional food photography",
    "cinematic food realism"
]

for i in range(1, NUM_IMAGES + 1):
    # 2. สุ่มเลือก Keyword จากแต่ละกลุ่ม
    selected_cat = random.choice(food_category)
    selected_action = random.choice(dish_type)
    selected_angle = random.choice(camera_angle)
    selected_face = random.choice(lighting)
    selected_style = random.choice(presentation)
    selected_light = random.choice(realism_style)
    
    # 3. สร้างเลข Seed แบบสุ่มจริงๆ (ใช้เลขหลักล้านเพื่อให้กระจายตัว)
    random_seed = random.randint(1, 9999999)
    
    # 4. ประกอบ Prompt ใหม่
    # ตัวอย่าง: "A tiger, side profile view, running, golden hour --seed 827342"
    prompt_full = f"{PROMPT_BASE}, {selected_cat}, {selected_action}, {selected_angle},{selected_face}, {selected_style}, {selected_light} --seed {random_seed}"
        
        # 3. สร้างชื่อไฟล์ตามรูปแบบ: seedream_people_Photorealistic_001.jpg
    file_name = f"seedream_food_Photorealistic_{str(i).zfill(3)}.jpg"
    output_path = os.path.join(OUTPUT_FOLDER, file_name)
        
    print(f"⏳ กำลังสร้างรูปภาพที่ {i}/{NUM_IMAGES}: '{file_name}'...")

    try:
            # 4. เรียกใช้ Model เพื่อสร้างรูปภาพ
            # Seedream API (ผ่าน OpenAI SDK) มักจะคืนค่าเป็น URL
            response = client.images.generate(
                model=MODEL_NAME,
                prompt=prompt_full, # ส่ง Prompt ที่มี seed
                n=1, # จำนวนรูปภาพต่อการเรียก API 1 ครั้ง
                size="3024x4032", # กำหนดขนาดรูปภาพ
                response_format="url" ,# ขอให้คืนค่าเป็น URL
                extra_body={"watermark": False})


            # 5. แยกและบันทึกรูปภาพ
            if response.data and response.data[0].url:
                image_url = response.data[0].url
                
                # ดาวน์โหลดรูปภาพจาก URL
                img_data = requests.get(image_url).content
                
                # บันทึกไฟล์
                with open(output_path, 'wb') as handler:
                    handler.write(img_data)
                    
                print(f"  ✨ บันทึกสำเร็จ: {output_path}")
            else:
                print("  ⚠️ API Response ไม่มี URL รูปภาพ")

    except Exception as e:
            print(f"  ❌ เกิดข้อผิดพลาดในการสร้าง/บันทึกรูปภาพที่ {i}: {e}")
            # หากเกิดข้อผิดพลาด ให้ข้ามไปสร้างรูปถัดไป

    print(f"\n🎉 การสร้างรูปภาพเสร็จสมบูรณ์ 50 รูป ถูกเก็บไว้ในโฟลเดอร์: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    # ตรวจสอบการติดตั้งไลบรารีที่จำเป็นก่อนรัน
    print("🔔 ตรวจสอบว่าคุณได้ติดตั้งไลบรารี: openai และ requests แล้ว (pip install openai requests)")
    create_and_save_images()

## Count the amount of pictures generated

In [ ]:
import os

image_dir = '/Users/rrinladarr/Desktop/AI_vs_Real_Project/test/images'
files = [f.lower() for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# นับแยกตาม Keyword ที่เราตั้งชื่อไว้
counts = {
    "Digital Art (Danbooru)": len([f for f in files if 'real_digital' in f]),
    "Animals": len([f for f in files if 'real_animal' in f]),
    "Human Profiles": len([f for f in files if 'real_human' in f]),
    "Food": len([f for f in files if 'real_food' in f]),
    "Places": len([f for f in files if 'real_place' in f]),
    "Objects (CIFAKE)": len([f for f in files if 'real_object' in f]),
    "AI (Gemini/Seedream)": len([f for f in files if any(k in f for k in ['gemini', 'seedream'])])
}

print(f"📊 สรุปยอดแยกตามหมวดหมู่ (เช็คหาตัวที่หายไป 1):")
print("-" * 40)
total = 0
for category, count in counts.items():
    print(f"{category:<25}: {count} รูป")
    total += count

print("-" * 40)
print(f"📈 รวมสุทธิ: {total} รูป")